In [1]:
import numpy as np
import tensorly as tl
from tensorly.decomposition import parafac

tl.set_backend('numpy')


def f(xyz, c=5):
    return 1 / (1 + c**2 * np.sum(xyz**2, axis=1))

def chebyshev_polys(x, deg):
    T = np.zeros((deg+1, len(x)))
    T[0] = 1
    if deg > 0:
        T[1] = x
    for k in range(2, deg+1):
        T[k] = 2 * x * T[k-1] - T[k-2]
    return T

def generate_coeff_tensor(N, c):
    k = np.arange(N+1)
    nodes = np.cos((2*k + 1) * np.pi / (2*(N+1)))
    X, Y, Z = np.meshgrid(nodes, nodes, nodes, indexing="ij")
    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    F = f(coords, c).reshape((N+1, N+1, N+1))

    T = chebyshev_polys(nodes, N)  # (N+1, N+1)

    # Raw projection (no normalization yet)
    C_raw = np.einsum('ia,jb,kc,ijk->abc', T, T, T, F)

    # Per-axis scaling vectors: s[0]=1/(N+1), s[n>=1]=2/(N+1)
    s = np.full(N+1, 2.0/(N+1))
    s[0] = 1.0/(N+1)

    # Apply along each mode (outer product of scalings)
    C = C_raw * s[:, None, None] * s[None, :, None] * s[None, None, :]

    return C, nodes


def evaluate_cp_interp(weights, factors, nodes, N, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    Tx = chebyshev_polys(xx, N)
    Ty = chebyshev_polys(yy, N)
    Tz = chebyshev_polys(zz, N)
    F_cp = np.zeros((resolution, resolution, resolution))

    for r in range(len(weights)):
        a = Tx.T @ factors[0][:, r]
        b = Ty.T @ factors[1][:, r]
        c = Tz.T @ factors[2][:, r]
        F_cp += weights[r] * np.einsum("i,j,k->ijk", a, b, c)

    return F_cp

def compute_exact_function_grid(f, c, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    X, Y, Z = np.meshgrid(xx, yy, zz, indexing="ij")
    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    F = f(coords, c).reshape((resolution, resolution, resolution))
    return F

def evaluate_direct_chebyshev_interp(C, nodes, N, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    Tx = chebyshev_polys(xx, N)
    Ty = chebyshev_polys(yy, N)
    Tz = chebyshev_polys(zz, N)
    F_interp = np.einsum("ia,jb,kc,abc->ijk", Tx.T, Ty.T, Tz.T, C)
    return F_interp

N = 64
c = 5
resolution = 100
ranks = range(1, 11)

C, nodes = generate_coeff_tensor(N, c)
F_true = compute_exact_function_grid(f, c, resolution)


ranks = range(1, 21)


for rank in ranks:
    cp_tensor = parafac(C, rank=rank, init='svd')
    weights, factors = cp_tensor.weights, cp_tensor.factors

    F_cp = evaluate_cp_interp(weights, factors, nodes, N, resolution)

    diff_func = F_true - F_cp
    rmse_func = np.sqrt(np.mean(diff_func**2))
    maxe_func = np.max(np.abs(diff_func))

    C_reconstructed = tl.cp_to_tensor(cp_tensor)
    l2_norm_coeff = np.linalg.norm(C - C_reconstructed)

    print(f"Rank {rank}: RMSE = {rmse_func:.2e}, MaxE = {maxe_func:.2e}, L2 Norm = {l2_norm_coeff:.2e}")

F_interp_baseline = evaluate_direct_chebyshev_interp(C, nodes, N, resolution)
diff_baseline = F_true - F_interp_baseline
rmse_baseline = np.sqrt(np.mean(diff_baseline**2))
maxe_baseline = np.max(np.abs(diff_baseline))

print(f"[Baseline] RMSE: {rmse_baseline:.2e}, MaxE: {maxe_baseline:.2e}")

Rank 1: RMSE = 7.60e-02, MaxE = 7.34e-01, L2 Norm = 2.07e-02
Rank 2: RMSE = 7.42e-02, MaxE = 7.12e-01, L2 Norm = 7.18e-03
Rank 3: RMSE = 7.42e-02, MaxE = 6.90e-01, L2 Norm = 1.56e-03
Rank 4: RMSE = 7.42e-02, MaxE = 6.84e-01, L2 Norm = 7.42e-04
Rank 5: RMSE = 7.42e-02, MaxE = 6.85e-01, L2 Norm = 4.51e-04
Rank 6: RMSE = 7.42e-02, MaxE = 6.86e-01, L2 Norm = 7.82e-04
Rank 7: RMSE = 7.42e-02, MaxE = 6.85e-01, L2 Norm = 4.76e-04
Rank 8: RMSE = 7.42e-02, MaxE = 6.84e-01, L2 Norm = 2.28e-04
Rank 9: RMSE = 7.42e-02, MaxE = 6.84e-01, L2 Norm = 2.41e-04
Rank 10: RMSE = 7.42e-02, MaxE = 6.84e-01, L2 Norm = 1.81e-04
Rank 11: RMSE = 7.42e-02, MaxE = 6.84e-01, L2 Norm = 1.72e-04
Rank 12: RMSE = 7.42e-02, MaxE = 6.84e-01, L2 Norm = 1.28e-04
Rank 13: RMSE = 7.42e-02, MaxE = 6.84e-01, L2 Norm = 1.26e-04
Rank 14: RMSE = 7.42e-02, MaxE = 6.83e-01, L2 Norm = 7.34e-05
Rank 15: RMSE = 7.42e-02, MaxE = 6.83e-01, L2 Norm = 3.82e-05
Rank 16: RMSE = 7.42e-02, MaxE = 6.83e-01, L2 Norm = 3.94e-05
Rank 17: RMSE = 7

In [2]:
import numpy as np
import tensorly as tl
from tensorly.decomposition import parafac

tl.set_backend('numpy')


def f(xyz, c=5):
    return 1 / (1 + c**2 * np.sum(xyz**2, axis=1))

def chebyshev_polys(x, deg):
    T = np.zeros((deg+1, len(x)))
    T[0] = 1
    if deg > 0:
        T[1] = x
    for k in range(2, deg+1):
        T[k] = 2 * x * T[k-1] - T[k-2]
    return T

def generate_coeff_tensor(N, c):
    # Chebyshev–Gauss nodes (roots)
    k = np.arange(N+1)
    nodes = np.cos((2*k + 1) * np.pi / (2*(N+1)))

    # Sample f on the tensor grid of nodes
    X, Y, Z = np.meshgrid(nodes, nodes, nodes, indexing="ij")
    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    F = f(coords, c).reshape((N+1, N+1, N+1))

    # 1D Chebyshev Vandermonde: T[n, i] = T_n(x_i)
    T = chebyshev_polys(nodes, N)                 # shape: (N+1, N+1)
    Tx = T.copy(); Ty = T.copy(); Tz = T.copy()

    # Build Kronecker collocation matrix A = (Tz^T ⊗ Ty^T ⊗ Tx^T)
    # Such that: A @ vec(C) ≈ vec(F)
    A = np.kron(Tz.T, np.kron(Ty.T, Tx.T))        # shape: ((N+1)^3, (N+1)^3)

    # Solve the (square) system in least squares sense
    F_flat = F.ravel()
    c_flat, *_ = np.linalg.lstsq(A, F_flat, rcond=None)

    # Reshape back to 3D Chebyshev coefficient tensor
    C = c_flat.reshape((N+1, N+1, N+1))
    return C, nodes

def evaluate_cp_interp(weights, factors, nodes, N, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    Tx = chebyshev_polys(xx, N)
    Ty = chebyshev_polys(yy, N)
    Tz = chebyshev_polys(zz, N)
    F_cp = np.zeros((resolution, resolution, resolution))

    for r in range(len(weights)):
        a = Tx.T @ factors[0][:, r]
        b = Ty.T @ factors[1][:, r]
        c = Tz.T @ factors[2][:, r]
        F_cp += weights[r] * np.einsum("i,j,k->ijk", a, b, c)

    return F_cp

def compute_exact_function_grid(f, c, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    X, Y, Z = np.meshgrid(xx, yy, zz, indexing="ij")
    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    F = f(coords, c).reshape((resolution, resolution, resolution))
    return F

# --- Params ---
N = 31
c = 5
resolution = 100

# --- Build coeffs and ground truth ---
C, nodes = generate_coeff_tensor(N, c)
F_true = compute_exact_function_grid(f, c, resolution)

# --- CP ranks 1..20, print RMSE only ---
for rank in range(1, 21):
    cp_tensor = parafac(C, rank=rank, init='svd')
    weights, factors = cp_tensor.weights, cp_tensor.factors
    F_cp = evaluate_cp_interp(weights, factors, nodes, N, resolution)

    rmse_func = np.sqrt(np.mean((F_true - F_cp)**2))
    print(f"Rank {rank}: RMSE = {rmse_func:.2e}")


Rank 1: RMSE = 2.39e-02
Rank 2: RMSE = 4.25e-03
Rank 3: RMSE = 8.67e-04
Rank 4: RMSE = 7.63e-04
Rank 5: RMSE = 2.74e-04
Rank 6: RMSE = 2.97e-04
Rank 7: RMSE = 2.28e-04
Rank 8: RMSE = 1.83e-04
Rank 9: RMSE = 1.53e-04
Rank 10: RMSE = 1.78e-04
Rank 11: RMSE = 1.68e-04
Rank 12: RMSE = 1.30e-04
Rank 13: RMSE = 1.18e-04
Rank 14: RMSE = 1.19e-04
Rank 15: RMSE = 1.12e-04
Rank 16: RMSE = 1.12e-04
Rank 17: RMSE = 1.11e-04
Rank 18: RMSE = 1.11e-04
Rank 19: RMSE = 1.10e-04
Rank 20: RMSE = 1.10e-04
